# 🚀 Unidad 1 — Tu Primer Agente de RL: LunarLander

**Curso: Aprendizaje por Refuerzo Profundo en Español**  
Adaptación pedagógica basada en el curso de HuggingFace 🤗  
Autor: Jose Miguel Lara Rangel · [aicraft.mx](https://aicraft.mx)

---

### ¿Qué haremos en este notebook?

Vamos a entrenar una **nave espacial** para que aprenda a aterrizar en la Luna — sin programarle explícitamente cómo hacerlo.  
El agente aprenderá solo, por prueba y error, usando el algoritmo **PPO** de la librería Stable-Baselines3.

Al terminar este notebook habrás:
- ✅ Explorado el entorno `LunarLander-v2` con Gymnasium
- ✅ Entendido qué observa y qué puede hacer la nave
- ✅ Entrenado tu primer agente de Deep RL
- ✅ Evaluado su desempeño con métricas reales
- ✅ Visualizado tu agente jugando en video

> 📎 **Slides de referencia:** Capítulo 1.5 del curso — revísalas antes si no lo has hecho.
> 
> ⏱️ **Tiempo estimado:** ~30 minutos (20 de entrenamiento, 10 de código)

---
## 0 · Primero lo primero: ¿qué vamos a lograr?

Antes de escribir una sola línea de código, veamos **el resultado final**:  
una nave que aprendió a aterrizar suavemente después de 1 millón de pasos de entrenamiento.

Observa:
- Cómo enciende los motores laterales para corregir la inclinación
- Cómo frena justo antes de tocar el suelo
- Cómo **no** fue programada para hacer nada de esto — lo descubrió sola

In [ ]:
%%html
<video controls autoplay loop style="width:100%; max-width:600px; border-radius:8px;">
  <source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4">
</video>
<p style="color:gray; font-size:0.85em;">Agente PPO entrenado en LunarLander-v2 · Modelo de Stable-Baselines3</p>

---
## 1 · Configurar Google Colab

### 1.1 · Activar la GPU (¡hazlo primero!)

El entrenamiento tarda **~20 minutos con GPU** y varias horas sin ella.  
Antes de correr cualquier celda, activa la GPU:

1. En el menú de Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución**
2. En *Acelerador de hardware* selecciona **GPU (T4)**
3. Haz clic en **Guardar**

> ⚠️ Si no haces esto primero y después lo cambias, Colab reiniciará el entorno y tendrás que volver a instalar todo.

### 1.2 · Instalar las dependencias

Necesitamos cuatro librerías principales:

| Librería | ¿Para qué sirve? |
|----------|-----------------|
| `stable-baselines3` | Implementaciones de algoritmos de RL (PPO, DQN, A2C...) |
| `gymnasium[box2d]` | El entorno LunarLander-v2 |
| `huggingface_sb3` | Puente entre SB3 y el Hub de HuggingFace |
| `swig` + `cmake` | Dependencias de compilación que necesita box2d |

La instalación puede tardar 2-3 minutos. Cuando termine verás `✅ Dependencias instaladas`.

In [ ]:
# Dependencias de compilación (necesarias para LunarLander)
!apt install -q swig cmake

# Librerías del curso
!pip install -q stable-baselines3==2.0.0a5 swig 'gymnasium[box2d]' huggingface_sb3

print("✅ Dependencias instaladas correctamente")

### 1.3 · Instalar herramientas para grabar video

Necesitamos una pantalla virtual para que el entorno pueda renderizar imágenes en Colab  
(Colab no tiene pantalla física, así que simulamos una).

> 📌 Después de esta celda, **reinicia el entorno de ejecución** cuando Colab lo solicite.
> Es normal — necesita que las nuevas librerías sean reconocidas.

In [ ]:
!sudo apt-get update -q
!sudo apt-get install -q -y python3-opengl
!apt install -q ffmpeg xvfb
!pip install -q pyvirtualdisplay

print("✅ Herramientas de video instaladas")
print("⚠️  Si Colab pide reiniciar el entorno de ejecución, acepta y vuelve a partir de la sección 1.4")

### 1.4 · Iniciar la pantalla virtual

Esta celda debe correrse **después de reiniciar** el entorno si Colab lo solicitó.

In [ ]:
from pyvirtualdisplay import Display

# Iniciar pantalla virtual (invisible) para renderizar el entorno
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

print("✅ Pantalla virtual iniciada")

---
## 2 · Importar las librerías

Vamos a importar todo lo que necesitamos. Aquí explicamos para qué sirve cada módulo:

In [ ]:
import gymnasium as gym                              # entornos de RL
import numpy as np                                   # operaciones numéricas

# Stable-Baselines3 — algoritmos de RL
from stable_baselines3 import PPO                    # el algoritmo que usaremos
from stable_baselines3.common.env_util import make_vec_env   # múltiples entornos en paralelo
from stable_baselines3.common.evaluation import evaluate_policy  # medir desempeño
from stable_baselines3.common.monitor import Monitor # registrar estadísticas del entorno

# Hugging Face — para visualizar y compartir modelos (opcional al final)
from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

print("✅ Librerías importadas")

---
## 3 · Conocer el entorno: LunarLander-v2

Antes de entrenar al agente, necesitamos entender en qué mundo vive.  
En RL esto se llama **explorar el entorno** — y es el primer paso en cualquier proyecto.

### 3.1 · Crear el entorno y hacer un reset

`gym.make()` crea una instancia del entorno.  
`env.reset()` reinicia el entorno al estado inicial y nos da la primera **observación**.

In [ ]:
# Crear el entorno
env = gym.make("LunarLander-v2")

# Reiniciar al estado inicial
observation, info = env.reset()

print("Entorno creado ✅")
print(f"Tipo de observación: {type(observation)}")
print(f"Valores de la primera observación: {observation}")

### 3.2 · ¿Qué información recibe la nave? (Espacio de observaciones)

La nave no ve la pantalla como nosotros. En cambio, recibe **8 números** que describen su situación actual:

| Índice | Descripción | Rango típico |
|--------|-------------|-------------|
| `obs[0]` | Posición horizontal (x) — 0 = centro | -1.5 a +1.5 |
| `obs[1]` | Posición vertical (y) — 0 = suelo | 0 a +1.5 |
| `obs[2]` | Velocidad horizontal | -5 a +5 |
| `obs[3]` | Velocidad vertical | -5 a +5 |
| `obs[4]` | Ángulo de inclinación | -π a +π |
| `obs[5]` | Velocidad angular | -5 a +5 |
| `obs[6]` | Pata izquierda toca el suelo | 0 (no) o 1 (sí) |
| `obs[7]` | Pata derecha toca el suelo | 0 (no) o 1 (sí) |

Estos 8 números son **todo lo que sabe el agente** en cada momento del tiempo.
No ve la pantalla, no sabe el color de nada — solo tiene estos valores.

In [ ]:
# Ver el espacio de observaciones
print("Forma del espacio de observaciones:", env.observation_space.shape)
print("Tipo:", env.observation_space.dtype)
print()

# Ver los valores mínimos y máximos posibles de cada variable
print("Valores mínimos posibles:")
print(env.observation_space.low)
print()
print("Valores máximos posibles:")
print(env.observation_space.high)

### 3.3 · ¿Qué puede hacer la nave? (Espacio de acciones)

El agente puede tomar **4 acciones discretas** en cada paso:

| Acción | Código | Descripción |
|--------|--------|-------------|
| No hacer nada | `0` | La nave cae libremente bajo la gravedad |
| Motor izquierdo | `1` | Empuja la nave hacia la derecha |
| Motor principal | `2` | Frena la caída (el más costoso en combustible) |
| Motor derecho | `3` | Empuja la nave hacia la izquierda |

**Discreto** significa que solo puede elegir una de esas 4 opciones — no puede activar el motor al 50%.

In [ ]:
print("Número de acciones posibles:", env.action_space.n)
print()

# Tomar una acción aleatoria y ver qué devuelve el entorno
action = env.action_space.sample()  # acción aleatoria
observation, reward, terminated, truncated, info = env.step(action)

print(f"Acción tomada: {action}")
print(f"Recompensa recibida: {reward:.4f}")
print(f"¿Episodio terminado?: {terminated}")
print(f"¿Tiempo agotado?: {truncated}")
print(f"Nueva observación: {observation}")

### 3.4 · El sistema de recompensas

La nave recibe puntos en cada paso según su comportamiento. Así aprende qué es bueno y qué es malo:

**Recompensas positivas ✅**
- Acercarse a la plataforma de aterrizaje
- Moverse lento (aterrizaje suave)
- Mantenerse horizontal (sin inclinación)
- **+10 puntos** por cada pata que toca el suelo
- **+100 puntos** por aterrizar exitosamente

**Penalizaciones ❌**
- **−0.3 puntos/frame** por usar el motor principal
- **−0.03 puntos/frame** por usar motores laterales
- **−100 puntos** por estrellarse

> 🎯 **Meta del curso:** Una recompensa media ≥ 200 indica que el agente aterrizó bien de forma consistente.
> Eso es lo que vamos a lograr.

### 3.5 · Ver el entorno en acción (agente aleatorio)

Antes de entrenar, veamos cómo se comporta un agente que **elige acciones al azar** — el punto de partida.

In [ ]:
# Simular 5 episodios con acciones completamente aleatorias
env_test = gym.make("LunarLander-v2")
recompensas_totales = []

for episodio in range(5):
    obs, _ = env_test.reset()
    recompensa_total = 0
    pasos = 0
    done = False

    while not done:
        action = env_test.action_space.sample()  # acción aleatoria
        obs, reward, terminated, truncated, _ = env_test.step(action)
        recompensa_total += reward
        pasos += 1
        done = terminated or truncated

    recompensas_totales.append(recompensa_total)
    print(f"Episodio {episodio+1}: recompensa = {recompensa_total:.1f} | pasos = {pasos}")

print()
print(f"Recompensa promedio (agente aleatorio): {np.mean(recompensas_totales):.1f}")
print("→ Después del entrenamiento esperamos superar 200 puntos de promedio")

---
## 4 · Preparar el entorno de entrenamiento

Para entrenar más eficientemente vamos a usar **16 copias del entorno en paralelo**.  
Imagínalo así: en lugar de tener una nave aprendiendo, tenemos 16 naves simultáneas,  
todas compartiendo lo que aprenden con el mismo modelo central.

Esto se llama **vectorización de entornos** y acelera el entrenamiento significativamente.

| Configuración | Pasos por segundo | Tiempo estimado (1M pasos) |
|--------------|------------------|--------------------------|
| 1 entorno | ~300 | ~55 min |
| 16 entornos | ~1,500+ | ~10-20 min |

In [ ]:
# Crear 16 copias del entorno en paralelo
env = make_vec_env("LunarLander-v2", n_envs=16)

print(f"Entornos en paralelo: 16")
print(f"Tipo de entorno vectorizado: {type(env)}")
print("✅ Entorno de entrenamiento listo")

---
## 5 · Definir el modelo PPO

Vamos a usar **PPO (Proximal Policy Optimization)**, uno de los algoritmos de RL más utilizados en la práctica.  
Lo estudiaremos a fondo en el Módulo 8 — por ahora lo usamos como una **caja negra potente**.

Lo que necesitas saber ahora:
- PPO es un algoritmo de **gradiente de política** (aprende directamente qué acciones tomar)
- `MlpPolicy` = red neuronal con capas densas, ideal cuando el estado son números (no imágenes)
- Si el estado fueran imágenes (como en Atari) usaríamos `CnnPolicy`

### Los hiperparámetros más importantes:

| Parámetro | Valor | ¿Qué controla? |
|-----------|-------|----------------|
| `n_steps` | 1024 | Pasos recolectados antes de cada actualización |
| `batch_size` | 64 | Tamaño del mini-lote para el gradiente |
| `n_epochs` | 4 | Veces que se reutiliza cada lote |
| `gamma` | 0.999 | Factor de descuento (casi sin descuento — valora el futuro casi igual que el presente) |
| `gae_lambda` | 0.98 | Controla el balance sesgo-varianza en la estimación de ventajas |
| `ent_coef` | 0.01 | Coeficiente de entropía — incentiva la exploración |

---
### 🏋️ EJERCICIO (opcional)

Antes de ver la solución, intenta definir el modelo tú mismo.

**Pistas:**
- El modelo es `PPO(...)`
- La política para observaciones numéricas es `'MlpPolicy'`
- El entorno ya está definido arriba como `env`
- Agrega `verbose=1` para ver el progreso durante el entrenamiento

Si prefieres no hacerlo como ejercicio, **salta esta celda** y ejecuta directamente la celda de Solución de abajo.

In [ ]:
# 🏋️ EJERCICIO — escribe tu código aquí
# model = PPO(
#     policy = ...,
#     env = ...,
#     n_steps = ...,
#     batch_size = ...,
#     n_epochs = ...,
#     gamma = ...,
#     gae_lambda = ...,
#     ent_coef = ...,
#     verbose = ...
# )

---
### ✅ Solución — ejecuta esta celda directamente si no hiciste el ejercicio

In [ ]:
# ✅ SOLUCIÓN
model = PPO(
    policy='MlpPolicy',   # red neuronal densa (para vectores numéricos)
    env=env,               # los 16 entornos en paralelo
    n_steps=1024,          # pasos por entorno antes de actualizar
    batch_size=64,         # tamaño del mini-lote
    n_epochs=4,            # pasadas por cada lote
    gamma=0.999,           # descuento (casi sin descuento)
    gae_lambda=0.98,       # balance sesgo-varianza
    ent_coef=0.01,         # incentivo a explorar
    verbose=1              # mostrar progreso en consola
)

print("✅ Modelo PPO definido")
print(f"Política: {model.policy}")

---
## 6 · Entrenar el agente

### ¿Qué significa '1,000,000 timesteps'?

Un **timestep** es un paso de simulación — la nave toma una acción y el entorno responde.  
Con 16 entornos en paralelo y 1 millón de pasos, el agente acumula una cantidad enorme de experiencia.

Durante el entrenamiento verás líneas como esta:
```
| rollout/ep_len_mean  | 234    |  → duración media de un episodio (pasos)
| rollout/ep_rew_mean  | -45.2  |  → recompensa media actual
| time/total_timesteps | 49152  |  → pasos totales completados
```

**Lo que hay que observar:** `ep_rew_mean` debe ir subiendo con el tiempo.  
Al principio será negativo (la nave se estrella). Al final debería superar **+200**.

> ☕ Este entrenamiento tarda **~20 minutos con GPU T4**.  
> Es completamente normal — el agente está aprendiendo miles de aterrizajes.

---
### 🏋️ EJERCICIO (opcional)

Llama a `model.learn()` con `total_timesteps=1_000_000`.
Luego guarda el modelo con `model.save('ppo-LunarLander-v2')`.

Si prefieres no hacerlo como ejercicio, salta a la celda de Solución.

In [ ]:
# 🏋️ EJERCICIO — escribe tu código aquí
# model.learn(...)
# model.save(...)

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
# Entrenar durante 1 millón de pasos (~20 min con GPU T4)
model.learn(total_timesteps=1_000_000)

# Guardar el modelo entrenado en disco
model_name = "ppo-LunarLander-v2"
model.save(model_name)

print(f"\n✅ Entrenamiento completado")
print(f"Modelo guardado como: {model_name}.zip")

---
## 7 · Evaluar el agente entrenado

Ahora medimos qué tan bien aterrizó el agente.  
**Regla importante:** evaluamos en un entorno *nuevo* — distinto al de entrenamiento.  
Esto evita el 'efecto trampa': que el agente haya memorizado ese entorno específico.

Usamos `Monitor` para registrar las estadísticas del episodio, y `evaluate_policy`  
para correr 10 episodios y calcular la recompensa media.

### ¿Cómo interpretar el resultado?

| Recompensa media | Interpretación |
|-----------------|----------------|
| < 0 | El agente se estrella frecuentemente |
| 0 – 100 | Aprende a no estrellarse, pero aterriza lejos |
| 100 – 200 | Aterriza cerca, aún con movimientos bruscos |
| **≥ 200** | **Meta del curso — aterrizaje exitoso y suave** ✅ |
| > 250 | Aterrizaje muy eficiente, usa poco combustible |

---
### 🏋️ EJERCICIO (opcional)

Crea el entorno de evaluación con `Monitor(gym.make(..., render_mode='rgb_array'))`,  
luego llama a `evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)`.

Si prefieres no hacerlo como ejercicio, salta a la celda de Solución.

In [ ]:
# 🏋️ EJERCICIO — escribe tu código aquí
# eval_env = ...
# mean_reward, std_reward = evaluate_policy(...)
# print(...)

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
# Crear entorno de evaluación (distinto al de entrenamiento)
eval_env = Monitor(gym.make("LunarLander-v2", render_mode="rgb_array"))

# Evaluar en 10 episodios con política determinista (sin aleatoriedad)
mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True  # sin exploración — el agente usa su mejor política
)

print(f"Recompensa media: {mean_reward:.2f} ± {std_reward:.2f}")

# Interpretar el resultado
if mean_reward >= 200:
    print("✅ ¡Meta alcanzada! Tu agente aterriza de forma consistente.")
elif mean_reward >= 100:
    print("🟡 Buen progreso. Prueba entrenar más pasos o ajustar gamma.")
else:
    print("🔴 El agente aún necesita más entrenamiento.")

---
## 8 · Ver al agente jugar en video

Los números son buenos, pero ver al agente en acción es mucho más motivador.  
Vamos a grabar un episodio completo y mostrarlo aquí en el notebook.

In [ ]:
import imageio
from IPython.display import Video, display
import os

def grabar_episodio(model, env_id, archivo_salida='replay.mp4', fps=30):
    """Graba un episodio completo del agente entrenado y lo guarda como video."""
    env_video = gym.make(env_id, render_mode="rgb_array")
    frames = []
    obs, _ = env_video.reset()
    done = False

    while not done:
        # El agente elige la acción (sin aleatoriedad)
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env_video.step(action)
        frames.append(env_video.render())
        done = terminated or truncated

    env_video.close()
    imageio.mimsave(archivo_salida, frames, fps=fps)
    return archivo_salida

# Grabar un episodio
print("Grabando episodio...")
video_path = grabar_episodio(model, "LunarLander-v2")
print(f"Video guardado: {video_path}")

# Mostrar el video en el notebook
display(Video(video_path, embed=True, width=500))

---
## 9 · Experimenta: cambia parámetros y observa

La mejor forma de entender los hiperparámetros es **cambiarlos y ver qué pasa**.  
Aquí te proponemos 3 experimentos rápidos. No hay respuesta correcta — la idea es observar.

> ⚠️ Cada experimento re-entrena el modelo desde cero (~20 min).  
> Puedes cambiar `total_timesteps=200_000` para ver tendencias rápidas sin esperar tanto.

### Experimento A · ¿Qué pasa con gamma más pequeño?

Un `gamma` más bajo hace que el agente valore **menos el futuro** y más la recompensa inmediata.  
¿Se vuelve más impaciente? ¿Aterrizará diferente?

In [ ]:
# Experimento A — gamma = 0.9 (más impaciente)
env_exp = make_vec_env("LunarLander-v2", n_envs=16)

model_exp_a = PPO(
    policy='MlpPolicy',
    env=env_exp,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.9,           # ← cambiado de 0.999 a 0.9
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1
)

# Para ver tendencias rápidas, usa 200_000 en lugar de 1_000_000
model_exp_a.learn(total_timesteps=200_000)

eval_env_exp = Monitor(gym.make("LunarLander-v2", render_mode="rgb_array"))
mean_a, std_a = evaluate_policy(model_exp_a, eval_env_exp, n_eval_episodes=10, deterministic=True)
print(f"\nExperimento A (gamma=0.9): {mean_a:.2f} ± {std_a:.2f}")
print(f"Modelo original  (gamma=0.999): {mean_reward:.2f} ± {std_reward:.2f}")

### Experimento B · ¿Cuánto importa el número de entornos en paralelo?

¿Qué diferencia hay entre entrenar con 1 entorno vs 16?  
Mismo tiempo de entrenamiento, distinta experiencia acumulada.

In [ ]:
# Experimento B — solo 1 entorno (sin paralelización)
env_exp_b = make_vec_env("LunarLander-v2", n_envs=1)

model_exp_b = PPO(
    policy='MlpPolicy',
    env=env_exp_b,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1
)

model_exp_b.learn(total_timesteps=200_000)

eval_env_b = Monitor(gym.make("LunarLander-v2", render_mode="rgb_array"))
mean_b, std_b = evaluate_policy(model_exp_b, eval_env_b, n_eval_episodes=10, deterministic=True)
print(f"\nExperimento B (1 entorno):  {mean_b:.2f} ± {std_b:.2f}")
print(f"Original  (16 entornos): {mean_reward:.2f} ± {std_reward:.2f}")

### Experimento C · Probar otro entorno con el mismo código

Una de las ventajas de SB3 es que el mismo código funciona en otros entornos.  
Prueba con `CartPole-v1` — equilibrar un palo sobre un carrito.  
Solo cambia el nombre del entorno y ajusta los pasos.

In [ ]:
# Experimento C — CartPole-v1 con el mismo código
env_cartpole = make_vec_env("CartPole-v1", n_envs=8)

model_cartpole = PPO(
    policy='MlpPolicy',
    env=env_cartpole,
    n_steps=512,
    batch_size=64,
    n_epochs=4,
    gamma=0.99,
    verbose=1
)

model_cartpole.learn(total_timesteps=100_000)

eval_cp = Monitor(gym.make("CartPole-v1"))
mean_cp, std_cp = evaluate_policy(model_cartpole, eval_cp, n_eval_episodes=10, deterministic=True)
print(f"\nCartPole-v1: recompensa media = {mean_cp:.2f} ± {std_cp:.2f}")
print("(meta de CartPole: ≥ 475 de 500 posibles)")

---

# 🔒 Secciones opcionales

> Las siguientes secciones requieren una cuenta de Hugging Face.  
> Son completamente opcionales — el objetivo principal del notebook ya está cumplido.
> Puedes saltarlas si prefieres.

---

## [OPCIONAL] Publicar tu agente en el Hugging Face Hub

Si tienes cuenta en HuggingFace, puedes subir tu modelo para que otros lo descarguen  
y vean tu nave aterrizando en el navegador. Para la certificación del curso de HF,  
necesitas una recompensa media ≥ 200.

### Paso 1: Iniciar sesión

1. Ve a [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Crea un token con permiso **Write**
3. Pégalo cuando la celda lo solicite

In [ ]:
from huggingface_hub import notebook_login

notebook_login()  # pegará tu token de acceso
!git config --global credential.helper store

### Paso 2: Subir el modelo al Hub

`package_to_hub` hace todo en un solo paso:
evalúa el modelo, graba el video replay, genera la tarjeta del modelo y lo sube al Hub.

In [ ]:
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv
from huggingface_sb3 import package_to_hub

# Cargar el modelo guardado
model_to_push = PPO.load("ppo-LunarLander-v2")

# Entorno de evaluación para generar el video
eval_env_hub = DummyVecEnv([lambda: Monitor(
    gym.make("LunarLander-v2", render_mode="rgb_array"))])

package_to_hub(
    model=model_to_push,
    model_name="ppo-LunarLander-v2",
    model_architecture="PPO",
    env_id="LunarLander-v2",
    eval_env=eval_env_hub,
    repo_id="TU_USUARIO/ppo-LunarLander-v2",  # ← cambia TU_USUARIO
    commit_message="Mi primer agente entrenado 🚀"
)

## [OPCIONAL] Cargar y evaluar un modelo del Hub

También puedes descargar modelos entrenados por la comunidad y evaluarlos.  
Esto es útil para comparar tu agente contra los mejores del leaderboard.

In [ ]:
# Necesitamos shimmy para compatibilidad con modelos entrenados en versiones anteriores de gym
!pip install -q shimmy

In [ ]:
from huggingface_sb3 import load_from_hub
from stable_baselines3.common.evaluation import evaluate_policy

# Descargar un modelo de la comunidad
repo_id = "Classroom-workshop/assignment2-omar"
filename = "ppo-LunarLander-v2.zip"

# Cargar el modelo
model_community = load_from_hub(repo_id=repo_id, filename=filename)
model_community = PPO.load(model_community)

# Evaluar
eval_community = Monitor(gym.make("LunarLander-v2", render_mode="rgb_array"))
mean_c, std_c = evaluate_policy(model_community, eval_community, n_eval_episodes=10, deterministic=True)
print(f"Modelo de la comunidad ({repo_id})")
print(f"Recompensa media: {mean_c:.2f} ± {std_c:.2f}")

---

## 🎉 ¡Notebook completado!

Acabas de entrenar tu primer agente de Deep Reinforcement Learning.  
Recapitulando lo que hiciste:

1. **Exploraste** el entorno `LunarLander-v2` — estados, acciones, recompensas
2. **Entrenaste** un agente PPO con Stable-Baselines3 en 1 millón de pasos
3. **Evaluaste** el desempeño con métricas estadísticas robustas
4. **Visualizaste** al agente jugando en video
5. **Experimentaste** cambiando hiperparámetros y observando el efecto

### ¿Qué sigue?

- 📓 **Notebook 2:** Q-Learning desde cero con NumPy — sin librerías mágicas
- 📊 **Capítulos 2.1–2.4:** La teoría detrás de Q-Learning, Bellman y MC vs TD

---
*Aicraft · aicraft.mx · Jose Miguel Lara Rangel*